# ASR Pipeline POC — diarization → separation → ASR

Drag-and-drop an audio file (mono or stereo, any sample rate). The notebook walks it through the full pipeline end-to-end:

```
Input ─► 1. Diarization (pyannote) ─► 2. Routing ─┬─► 3a. SE (MP-SENet @ 16 kHz) ──┐
                                                  └─► 3b. SS (SepFormer @ 8 kHz) + VAD ─┴─► 4. Assembly ─► 5. ASR (Whisper) ─► 6. SQUIM
```

Each stage has its own accordion in the Gradio UI with visualizations and audio playback.

## Prerequisites (one-time)

```bash
# From the repo's venv
pip install pyannote.audio speechbrain openai-whisper jiwer gradio soundfile

# Clone MP-SENet and download its VoiceBank+DEMAND checkpoint
git clone https://github.com/yxlu-0102/MP-SENet /home/user/MP-SENet
# Then download `best_ckpt/g_best` into that directory (see the MP-SENet README).
```

Gated models — accept the user agreement on huggingface.co/ for:
- `pyannote/speaker-diarization-3.1`
- `pyannote/segmentation-3.0`

Set your HF token and MP-SENet path in the config cell below.

## Scope note — v1 POC

Stage 6 is SQUIM non-intrusive metrics only (STOI, PESQ, SI-SDR) on mixture vs. per-speaker assembled streams. Intrusive metrics (cpWER, pWER, DER, overlap-F1) will be added in a follow-up once CLARIN tier annotations land.

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────

HF_TOKEN = "hf_YOUR_TOKEN_HERE"  # same token as asr/diarization.ipynb

# SepFormer separation checkpoint (PolSESS SB @ 8 kHz)
SEPFORMER_CKPT = "checkpoints/sepformer/SB/summer-capybara-228/sepformer_SB_best.pt"

# Path to the cloned MP-SENet repository
MP_SENET_ROOT = "/home/user/MP-SENet"
MP_SENET_CKPT = "best_ckpt/g_best_vb"          # relative to MP_SENET_ROOT

# Whisper
WHISPER_MODEL = "large-v3"
WHISPER_INITIAL_PROMPT = "Rozmowa po polsku."  # discourages language drift on noisy tails

# Routing defaults — exposed as sliders in the UI
MIN_OVERLAP_DUR     = 0.20   # drop overlaps shorter than this (backchannel filter)
MAX_MERGE_GAP       = 0.10   # merge same-speaker segments with gaps smaller than this
SEGMENT_PAD         = 0.10   # symmetric padding applied to each segment
VAD_THRESHOLD       = 0.50   # silero-vad probability threshold
MIN_SOLO_FOR_ANCHOR = 3.0    # weak-anchor fallback threshold (seconds of clean solo per speaker)

# Sample rates
SR_PIPELINE  = 16_000        # working rate for diarization / SE / VAD / ASR / SQUIM
SR_SEPARATOR = 8_000         # SepFormer was trained at 8 kHz

In [2]:
# ── Imports ───────────────────────────────────────────────────────────────────

import sys, os, json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torchaudio
import torchaudio.functional as AF

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Resolve project root and put it on sys.path (the notebook may be launched
# from either the repo root or asr/).
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "asr":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from utils.model_utils import load_model_for_inference  # noqa: E402

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
print(f"Project root: {PROJECT_ROOT}")

Device: cuda
Project root: /home/user/polsess_separation


## Model loading (one-time)

All six models are loaded at module scope so the Gradio callback is fast. Each block is independent — if MP-SENet isn't installed yet, comment out just that block and Stage 3a will skip enhancement.

In [ ]:
# ── GPU memory strategy ──────────────────────────────────────────────────────
# All six models loaded together exceed our 12 GB GPU. Strategy: load every
# model on CPU at module scope; move only the model needed for the current
# stage to GPU, then back to CPU before the next stage. Tensor ops keep
# placing new tensors on `DEVICE`; only model weights are shuffled.
LOAD_DEVICE = torch.device("cpu") if DEVICE.type == "cuda" else DEVICE


def _move(model, device):
    """Move pyannote.Pipeline / speechbrain.Pretrained / nn.Module to a device.

    Also updates `model.device` when it is a regular instance attribute
    (speechbrain Pretrained sets it once in __init__ and does not refresh on
    `.to()`, which causes encode_batch to ship inputs back to the load device
    and trigger CUDA/CPU weight mismatches).
    """
    if model is None:
        return None
    dev = device if isinstance(device, torch.device) else torch.device(device)
    try:
        model.to(dev)
    except TypeError:
        model.to(str(dev))
    # Only overwrite if `device` is a real instance attribute (not a property
    # like Whisper's). vars(model) only contains the instance __dict__.
    if "device" in vars(model):
        try:
            model.device = dev
        except (AttributeError, TypeError):
            pass
    return model


class on_gpu:
    """Context manager: model.to(GPU) on enter, model.to(CPU)+empty_cache on exit."""

    def __init__(self, *models):
        self.models = [m for m in models if m is not None]

    def __enter__(self):
        for m in self.models:
            _move(m, DEVICE)
        return self.models[0] if len(self.models) == 1 else tuple(self.models)

    def __exit__(self, exc_type, exc, tb):
        for m in self.models:
            _move(m, LOAD_DEVICE)
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()
        return False


print(f"Load device: {LOAD_DEVICE} | run device: {DEVICE}")

In [ ]:
# ── pyannote speaker-diarization-3.1 ─────────────────────────────────────────
from pyannote.audio import Pipeline

pyannote_pipe = Pipeline.from_pretrained(
    "pyannote/speaker-diarization-3.1", token=HF_TOKEN
).to(LOAD_DEVICE)
print("pyannote: loaded")

In [ ]:
# ── SepFormer separator (PolSESS SB @ 8 kHz) ─────────────────────────────────
separator, _sep_ckpt = load_model_for_inference(
    str(PROJECT_ROOT / SEPFORMER_CKPT), device=str(LOAD_DEVICE)
)
separator.eval()
print(f"SepFormer: loaded (epoch {_sep_ckpt.get('epoch', '?')}, "
      f"val SI-SDR {_sep_ckpt.get('val_sisdr', float('nan')):.2f} dB)")

In [ ]:
# ── MP-SENet (speech enhancement @ 16 kHz) ───────────────────────────────────
# MP-SENet ships as a standalone repo with top-level `models/` and `utils.py`
# that clash with this project's namespaces. We evict the project's modules,
# explicitly register MP-SENet's `models` package, load MPNet while that
# namespace is active, then restore the project's modules.

class _AttrDict(dict):
    def __getattr__(self, name):
        try:
            return self[name]
        except KeyError as e:
            raise AttributeError(name) from e


_NAMESPACES_TO_SWAP = ("models", "utils")


def _swap_out(prefixes):
    saved = {}
    for k in list(sys.modules):
        if k in prefixes or any(k.startswith(p + ".") for p in prefixes):
            saved[k] = sys.modules.pop(k)
    return saved


def _swap_in(saved, prefixes):
    for k in list(sys.modules):
        if k in prefixes or any(k.startswith(p + ".") for p in prefixes):
            del sys.modules[k]
    sys.modules.update(saved)


def _load_mpsenet(root: Path, ckpt_rel: str, device):
    import importlib, types
    root = Path(root).resolve()

    sys.path.insert(0, str(root))
    saved = _swap_out(_NAMESPACES_TO_SWAP)
    try:
        importlib.invalidate_caches()
        # This project's models/ is a regular package (__init__.py exists).
        # Python's path-importer cache remembers it even after we evict the
        # module from sys.modules, so `import models.model` still resolves to
        # the project dir.  Fix: explicitly create a package module whose
        # __path__ points at MP-SENet's models/ directory.
        models_pkg = types.ModuleType("models")
        models_pkg.__path__ = [str(root / "models")]
        sys.modules["models"] = models_pkg

        _mp_gen = importlib.import_module("models.model")
        MPNet = _mp_gen.MPNet
        with open(root / "config.json") as f:
            h = _AttrDict(json.load(f))
        model = MPNet(h).to(device)
        state = torch.load(root / ckpt_rel, map_location=device)
        model.load_state_dict(state["generator"])
        model.eval()
    finally:
        _swap_in(saved, _NAMESPACES_TO_SWAP)
        if str(root) in sys.path:
            sys.path.remove(str(root))

    return model, h


mpsenet, mpsenet_h = _load_mpsenet(MP_SENET_ROOT, MP_SENET_CKPT, LOAD_DEVICE)
print("MP-SENet: loaded")

In [ ]:
# ── Silero-VAD ───────────────────────────────────────────────────────────────
vad_model, vad_utils = torch.hub.load(
    "snakers4/silero-vad", "silero_vad", trust_repo=True
)
vad_model = vad_model.to(LOAD_DEVICE)
print("silero-vad: loaded")

# ── ECAPA-TDNN speaker embeddings ────────────────────────────────────────────
from speechbrain.inference.speaker import EncoderClassifier

ecapa = EncoderClassifier.from_hparams(
    source="speechbrain/spkrec-ecapa-voxceleb",
    run_opts={"device": str(LOAD_DEVICE)},
    savedir=str(PROJECT_ROOT / ".cache/ecapa"),
)
ecapa.eval()
print("ECAPA-TDNN: loaded")

# ── Whisper ──────────────────────────────────────────────────────────────────
import whisper
whisper_model = whisper.load_model(WHISPER_MODEL, device=str(LOAD_DEVICE))
print(f"Whisper {WHISPER_MODEL}: loaded")

# ── SQUIM non-intrusive quality predictor ────────────────────────────────────
from torchaudio.pipelines import SQUIM_OBJECTIVE
squim_obj = SQUIM_OBJECTIVE.get_model().to(LOAD_DEVICE)
squim_obj.eval()
print("SQUIM: loaded")

## Stage 1 — Diarization

Runs pyannote on mono-16 kHz audio, forcing two speakers. Returns:
- `segments_df` — one row per `(start, end, speaker)` turn
- `overlaps_df` — timeline of regions where `num_active_speakers ≥ 2`
- `total_duration` — in seconds

Visualization: timeline plot with per-speaker bars and overlap highlights (adapted from `asr/diarization.ipynb`).

In [ ]:
def load_audio_as_mono_16k(audio_path: str) -> np.ndarray:
    """Load any audio file, downmix to mono, resample to 16 kHz, return 1D float32."""
    waveform, sr = torchaudio.load(audio_path)
    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)
    if sr != SR_PIPELINE:
        waveform = AF.resample(waveform, sr, SR_PIPELINE)
    return waveform.squeeze(0).numpy().astype(np.float32)


def diarize(audio_16k: np.ndarray):
    """Run pyannote on mono 16 kHz audio. Returns (segments_df, overlaps_df, total_dur)."""
    waveform = torch.from_numpy(audio_16k).unsqueeze(0)
    with on_gpu(pyannote_pipe):
        result = pyannote_pipe(
            {"waveform": waveform, "sample_rate": SR_PIPELINE},
            num_speakers=2,
        )
    diar = result.speaker_diarization if hasattr(result, "speaker_diarization") else result

    seg_records = [
        {"start": round(t.start, 3), "end": round(t.end, 3),
         "duration": round(t.duration, 3), "speaker": spk}
        for t, _, spk in diar.itertracks(yield_label=True)
    ]
    seg_df = pd.DataFrame(seg_records)

    ovl_records = [
        {"start": round(s.start, 3), "end": round(s.end, 3),
         "duration": round(s.duration, 3)}
        for s in diar.get_overlap()
    ]
    ovl_df = pd.DataFrame(ovl_records) if ovl_records else pd.DataFrame(
        columns=["start", "end", "duration"]
    )

    total_dur = len(audio_16k) / SR_PIPELINE
    return seg_df, ovl_df, total_dur


def plot_diarization(seg_df, ovl_df, total_dur, title="Diarization"):
    """Timeline plot with speaker bars + overlap highlights. Returns a Figure."""
    if len(seg_df) == 0:
        fig, ax = plt.subplots(figsize=(12, 2))
        ax.text(0.5, 0.5, "No speakers detected", ha="center", va="center")
        return fig

    speakers = sorted(seg_df["speaker"].unique())
    spk_to_y = {s: i for i, s in enumerate(speakers)}
    cmap = plt.cm.get_cmap("Set2", max(len(speakers), 3))
    spk_color = {s: cmap(i) for i, s in enumerate(speakers)}

    fig, ax = plt.subplots(figsize=(16, max(2.5, len(speakers) * 1.0 + 1.0)))
    for _, row in seg_df.iterrows():
        ax.barh(spk_to_y[row["speaker"]], row["duration"], left=row["start"],
                height=0.6, color=spk_color[row["speaker"]], alpha=0.85)
    for _, row in ovl_df.iterrows():
        ax.axvspan(row["start"], row["end"], color="#e74c3c", alpha=0.25, zorder=0)

    ax.set_yticks(range(len(speakers)))
    ax.set_yticklabels(speakers, fontweight="bold")
    ax.set_xlabel("Time (s)")
    ax.set_xlim(0, total_dur)
    ax.set_ylim(-0.5, len(speakers) - 0.5)
    ax.invert_yaxis()
    ax.set_title(title, fontweight="bold")
    ax.grid(axis="x", alpha=0.3, linestyle="--")

    handles = [mpatches.Patch(color=spk_color[s], label=s) for s in speakers]
    handles.append(mpatches.Patch(color="#e74c3c", alpha=0.4, label="overlap"))
    ax.legend(handles=handles, loc="upper right")
    plt.tight_layout()
    return fig

## Stage 2 — Routing

Partitions the timeline into `{silence, solo-A, solo-B, overlap}` regions:

1. Drop overlap segments with `duration < MIN_OVERLAP_DUR` (filters backchannels).
2. Merge consecutive same-speaker segments with gaps smaller than `MAX_MERGE_GAP`.
3. Pad every surviving segment by `SEGMENT_PAD` on both sides.
4. Per speaker, `solo = speaker_regions − all_overlap_regions`.
5. Whatever's uncovered becomes `silence`.

Output is a single `partition_df` ordered by start time.

In [ ]:
def _merge_consecutive(segs, max_gap):
    """Merge consecutive (s, e) intervals whose gap is below max_gap."""
    if not segs:
        return []
    segs = sorted(segs)
    out = [list(segs[0])]
    for s, e in segs[1:]:
        if s - out[-1][1] < max_gap:
            out[-1][1] = max(out[-1][1], e)
        else:
            out.append([s, e])
    return [tuple(x) for x in out]


def _subtract(regions_a, regions_b):
    """regions_a minus regions_b. Both are lists of (s, e)."""
    result = []
    regions_b = sorted(regions_b)
    for s, e in sorted(regions_a):
        current = [(s, e)]
        for bs, be in regions_b:
            next_cur = []
            for cs, ce in current:
                if be <= cs or bs >= ce:
                    next_cur.append((cs, ce))
                    continue
                if bs > cs:
                    next_cur.append((cs, bs))
                if be < ce:
                    next_cur.append((be, ce))
            current = next_cur
        result.extend(current)
    return [(s, e) for s, e in result if e - s > 1e-3]


def route(seg_df, ovl_df, total_dur, min_overlap, max_gap, pad):
    """Partition the timeline into {silence, solo-A, solo-B, overlap}."""
    if len(seg_df) == 0:
        return pd.DataFrame(columns=["start", "end", "duration", "kind", "speaker"]), []

    ovl_raw = [(r.start, r.end) for r in ovl_df.itertuples() if r.duration >= min_overlap]
    ovl_padded = _merge_consecutive(
        [(max(0.0, s - pad), min(total_dur, e + pad)) for s, e in ovl_raw], 1e-6
    )

    speakers = sorted(seg_df["speaker"].unique())
    spk_regions = {}
    for spk in speakers:
        segs = [(r.start, r.end) for r in seg_df[seg_df["speaker"] == spk].itertuples()]
        segs = _merge_consecutive(segs, max_gap)
        segs = [(max(0.0, s - pad), min(total_dur, e + pad)) for s, e in segs]
        segs = _merge_consecutive(segs, 1e-6)
        spk_regions[spk] = segs

    partition = []
    for i, spk in enumerate(speakers):
        tag = f"solo-{chr(ord('A') + i)}"
        solos = _subtract(spk_regions[spk], ovl_padded)
        for s, e in solos:
            partition.append({"start": s, "end": e, "duration": e - s,
                              "kind": tag, "speaker": spk})
    for s, e in ovl_padded:
        partition.append({"start": s, "end": e, "duration": e - s,
                          "kind": "overlap", "speaker": None})

    active = _merge_consecutive([(r["start"], r["end"]) for r in partition], 1e-6)
    for s, e in _subtract([(0.0, total_dur)], active):
        partition.append({"start": s, "end": e, "duration": e - s,
                          "kind": "silence", "speaker": None})

    part_df = pd.DataFrame(partition).sort_values("start").reset_index(drop=True)
    return part_df, speakers


def plot_routing(seg_df, ovl_df, part_df, total_dur):
    """Before/after: top = raw diarization, bottom = partitioned timeline."""
    fig, (ax_raw, ax_routed) = plt.subplots(2, 1, figsize=(16, 5), sharex=True)

    speakers = sorted(seg_df["speaker"].unique()) if len(seg_df) else []
    spk_to_y = {s: i for i, s in enumerate(speakers)}
    cmap = plt.cm.get_cmap("Set2", max(len(speakers), 3))
    spk_color = {s: cmap(i) for i, s in enumerate(speakers)}
    for _, row in seg_df.iterrows():
        ax_raw.barh(spk_to_y[row["speaker"]], row["duration"], left=row["start"],
                    height=0.6, color=spk_color[row["speaker"]], alpha=0.85)
    for _, row in ovl_df.iterrows():
        ax_raw.axvspan(row["start"], row["end"], color="#e74c3c", alpha=0.25, zorder=0)
    ax_raw.set_yticks(range(len(speakers)))
    ax_raw.set_yticklabels(speakers)
    ax_raw.invert_yaxis()
    ax_raw.set_title("Raw diarization")

    kind_colors = {"silence": "#cccccc", "solo-A": "#1f77b4",
                   "solo-B": "#2ca02c", "overlap": "#e74c3c"}
    kind_y = {"silence": 0, "solo-A": 1, "solo-B": 2, "overlap": 3}
    for _, row in part_df.iterrows():
        ax_routed.barh(kind_y[row["kind"]], row["duration"], left=row["start"],
                       height=0.6, color=kind_colors[row["kind"]], alpha=0.85)
    ax_routed.set_yticks([0, 1, 2, 3])
    ax_routed.set_yticklabels(["silence", "solo-A", "solo-B", "overlap"])
    ax_routed.invert_yaxis()
    ax_routed.set_title("Routed partition")
    ax_routed.set_xlabel("Time (s)")
    ax_routed.set_xlim(0, total_dur)

    for ax in (ax_raw, ax_routed):
        ax.grid(axis="x", alpha=0.3, linestyle="--")
    plt.tight_layout()
    return fig

## Stage 3a — SE on solo regions (MP-SENet, 16 kHz)

Each solo-A / solo-B region from the partition is sliced out of the mono-16 kHz audio, passed through MP-SENet, and tagged with its speaker. The STFT pre-/post-processing follows MP-SENet's `utils.mag_pha_stft` / `mag_pha_istft` exactly (compressed magnitude, separate phase).

In [ ]:
def _mpsenet_stft(audio_t, h):
    """Match MP-SENet preprocessing: compressed-magnitude + unwrapped-phase."""
    window = torch.hann_window(h["win_size"]).to(audio_t.device)
    spec = torch.stft(
        audio_t, h["n_fft"], hop_length=h["hop_size"], win_length=h["win_size"],
        window=window, center=True, pad_mode="reflect",
        normalized=False, return_complex=True,
    )
    mag = spec.abs().pow(h["compress_factor"])
    pha = spec.angle()
    return mag, pha


def _mpsenet_istft(mag, pha, h):
    mag = mag.pow(1.0 / h["compress_factor"])
    spec = torch.complex(mag * torch.cos(pha), mag * torch.sin(pha))
    window = torch.hann_window(h["win_size"]).to(spec.device)
    return torch.istft(
        spec, h["n_fft"], hop_length=h["hop_size"], win_length=h["win_size"],
        window=window, center=True, normalized=False, onesided=True,
    )


@torch.no_grad()
def mpsenet_enhance(audio_np_16k: np.ndarray) -> np.ndarray:
    """Enhance a 1D float32 numpy array at 16 kHz. Caller is responsible for
    ensuring `mpsenet` is on DEVICE (use `with on_gpu(mpsenet):`)."""
    if len(audio_np_16k) < mpsenet_h["win_size"]:
        return audio_np_16k.copy()
    audio = torch.from_numpy(audio_np_16k).to(DEVICE)
    norm = torch.sqrt(audio.numel() / (audio.pow(2).sum() + 1e-12))
    audio_n = (audio * norm).unsqueeze(0)
    mag, pha = _mpsenet_stft(audio_n, mpsenet_h)
    out = mpsenet(mag, pha)
    if isinstance(out, tuple) or (isinstance(out, list)):
        amp_out, pha_out = out[0], out[1]
    else:
        amp_out, pha_out = out, pha
    audio_out = _mpsenet_istft(amp_out, pha_out, mpsenet_h).squeeze(0)
    audio_out = (audio_out / (norm + 1e-12)).cpu().numpy().astype(np.float32)
    if len(audio_out) < len(audio_np_16k):
        audio_out = np.pad(audio_out, (0, len(audio_np_16k) - len(audio_out)))
    else:
        audio_out = audio_out[: len(audio_np_16k)]
    return audio_out


def enhance_solo(audio_16k, part_df):
    """Run MP-SENet on every solo-X region."""
    results = {}
    solo_rows = part_df[part_df["kind"].str.startswith("solo")]
    if len(solo_rows) == 0:
        return results
    with on_gpu(mpsenet):
        for idx, row in solo_rows.iterrows():
            s_idx = int(row["start"] * SR_PIPELINE)
            e_idx = int(row["end"] * SR_PIPELINE)
            raw = audio_16k[s_idx:e_idx].astype(np.float32)
            enhanced = mpsenet_enhance(raw) if len(raw) >= 256 else raw
            key = f"{row['kind']}_#{idx}"
            results[key] = {
                "key": key,
                "start": row["start"], "end": row["end"],
                "speaker": row["speaker"],
                "raw": raw, "enhanced": enhanced,
            }
    return results


def plot_se_segment(seg: dict):
    """Waveform + spectrogram before/after for one segment."""
    fig, axes = plt.subplots(2, 2, figsize=(14, 6))
    t = np.arange(len(seg["raw"])) / SR_PIPELINE
    axes[0, 0].plot(t, seg["raw"], linewidth=0.5, color="C0")
    axes[0, 0].set_title("Raw waveform")
    axes[0, 1].plot(t, seg["enhanced"], linewidth=0.5, color="C1")
    axes[0, 1].set_title("Enhanced waveform")
    for ax in axes[0]:
        ax.set_xlabel("Time (s)")
        ax.grid(alpha=0.3)
    if len(seg["raw"]) > 256:
        axes[1, 0].specgram(seg["raw"], NFFT=512, Fs=SR_PIPELINE, noverlap=256)
        axes[1, 1].specgram(seg["enhanced"], NFFT=512, Fs=SR_PIPELINE, noverlap=256)
    axes[1, 0].set_title("Raw spectrogram")
    axes[1, 1].set_title("Enhanced spectrogram")
    for ax in axes[1]:
        ax.set_xlabel("Time (s)")
        ax.set_ylabel("Hz")
    plt.suptitle(f"{seg['key']} — {seg['start']:.2f}s → {seg['end']:.2f}s  ({seg['speaker']})")
    plt.tight_layout()
    return fig

## Stage 3b — SS on overlap regions + VAD gate

For each overlap region:
1. Resample 16 kHz → 8 kHz.
2. Run the SepFormer separator (trained at 8 kHz on PolSESS SB).
3. Resample each of the 2 output streams back to 16 kHz.
4. Gate each stream with Silero-VAD (zero out frames where P(speech) < threshold) to suppress separation residuals before Whisper sees them.

In [ ]:
_VAD_WINDOW = 512  # silero-vad expects exactly 512 samples per window at 16 kHz


def _vad_mask_16k(audio_16k: np.ndarray, threshold: float) -> np.ndarray:
    """Frame-level VAD mask at 16 kHz. Caller ensures vad_model is on DEVICE."""
    audio = torch.from_numpy(audio_16k).to(DEVICE)
    n = audio.numel()
    if n < _VAD_WINDOW:
        return np.ones(n, dtype=np.float32)
    n_windows = n // _VAD_WINDOW
    probs = np.zeros(n_windows, dtype=np.float32)
    vad_model.reset_states()
    with torch.no_grad():
        for i in range(n_windows):
            chunk = audio[i * _VAD_WINDOW:(i + 1) * _VAD_WINDOW]
            p = vad_model(chunk.unsqueeze(0), SR_PIPELINE)
            probs[i] = p.item() if hasattr(p, "item") else float(p)
    mask_frames = (probs > threshold).astype(np.float32)
    full_mask = np.zeros(n, dtype=np.float32)
    for i, m in enumerate(mask_frames):
        full_mask[i * _VAD_WINDOW:(i + 1) * _VAD_WINDOW] = m
    full_mask[n_windows * _VAD_WINDOW:] = 1.0
    return full_mask


@torch.no_grad()
def separate_overlap(audio_16k_seg: np.ndarray):
    """SepFormer on one overlap segment. Caller ensures separator is on DEVICE."""
    orig_len = len(audio_16k_seg)
    audio_t = torch.from_numpy(audio_16k_seg).unsqueeze(0)
    audio_8k = AF.resample(audio_t, SR_PIPELINE, SR_SEPARATOR).to(DEVICE)
    est = separator(audio_8k)  # [1, 2, T8]
    s1_8k = est[:, 0, :].cpu()
    s2_8k = est[:, 1, :].cpu()
    s1_16k = AF.resample(s1_8k, SR_SEPARATOR, SR_PIPELINE).squeeze(0).numpy().astype(np.float32)
    s2_16k = AF.resample(s2_8k, SR_SEPARATOR, SR_PIPELINE).squeeze(0).numpy().astype(np.float32)

    def _fix(arr):
        if len(arr) < orig_len:
            return np.pad(arr, (0, orig_len - len(arr)))
        return arr[:orig_len]

    return _fix(s1_16k), _fix(s2_16k)


def separate_and_vad(audio_16k, part_df, vad_threshold):
    """Run separation on overlap regions, then VAD-gate each pair of streams.

    Phases are run sequentially: separator on GPU first (large model), then
    swapped out before VAD goes on GPU.  Cached per-overlap mixtures and raw
    streams live on CPU between phases.
    """
    overlap_rows = list(part_df[part_df["kind"] == "overlap"].iterrows())
    if not overlap_rows:
        return []

    # Phase 1 — separation
    raw_pairs = []
    with on_gpu(separator):
        for idx, row in overlap_rows:
            s_idx = int(row["start"] * SR_PIPELINE)
            e_idx = int(row["end"] * SR_PIPELINE)
            mix = audio_16k[s_idx:e_idx].astype(np.float32)
            if len(mix) < 256:
                continue
            s1, s2 = separate_overlap(mix)
            raw_pairs.append({
                "idx": idx, "start": row["start"], "end": row["end"],
                "mix": mix, "s1_raw": s1, "s2_raw": s2,
            })

    # Phase 2 — VAD
    results = []
    with on_gpu(vad_model):
        for r in raw_pairs:
            mask1 = _vad_mask_16k(r["s1_raw"], vad_threshold)
            mask2 = _vad_mask_16k(r["s2_raw"], vad_threshold)
            results.append({
                **r,
                "s1_gated": r["s1_raw"] * mask1, "mask1": mask1,
                "s2_gated": r["s2_raw"] * mask2, "mask2": mask2,
            })
    return results


def plot_overlap_segment(seg: dict):
    """Mixture + stream 1/2 (raw vs gated) + VAD masks."""
    fig, axes = plt.subplots(4, 1, figsize=(14, 7), sharex=True)
    t = np.arange(len(seg["mix"])) / SR_PIPELINE
    axes[0].plot(t, seg["mix"], linewidth=0.5, color="C0"); axes[0].set_ylabel("mixture")
    axes[1].plot(t, seg["s1_raw"], linewidth=0.5, color="C1", alpha=0.4, label="raw")
    axes[1].plot(t, seg["s1_gated"], linewidth=0.5, color="C1", label="gated")
    axes[1].fill_between(t, -1, 1, where=(seg["mask1"] < 0.5),
                         color="gray", alpha=0.15, step="post")
    axes[1].set_ylabel("stream 1"); axes[1].legend(loc="upper right", fontsize=8)
    axes[2].plot(t, seg["s2_raw"], linewidth=0.5, color="C2", alpha=0.4, label="raw")
    axes[2].plot(t, seg["s2_gated"], linewidth=0.5, color="C2", label="gated")
    axes[2].fill_between(t, -1, 1, where=(seg["mask2"] < 0.5),
                         color="gray", alpha=0.15, step="post")
    axes[2].set_ylabel("stream 2"); axes[2].legend(loc="upper right", fontsize=8)
    axes[3].plot(t, seg["mask1"], color="C1", label="mask 1")
    axes[3].plot(t, seg["mask2"], color="C2", label="mask 2", alpha=0.7)
    axes[3].set_ylabel("VAD mask"); axes[3].set_xlabel("Time (s)")
    axes[3].legend(loc="upper right", fontsize=8); axes[3].set_ylim(-0.1, 1.1)
    for ax in axes:
        ax.grid(alpha=0.3)
    plt.suptitle(f"Overlap #{seg['idx']} — {seg['start']:.2f}s → {seg['end']:.2f}s")
    plt.tight_layout()
    return fig

## Stage 4 — Stream assembly

1. Build a speaker anchor for each diarized speaker: concatenate all their (enhanced) solo audio and compute one ECAPA-TDNN embedding.
2. **Weak-anchor check**: if either speaker has less than `MIN_SOLO_FOR_ANCHOR` seconds of solo material, flag `weak_anchor=True`. (Anchor-based assignment is unreliable; a later follow-up will gate pWER on this flag.)
3. For each overlap region, embed both separated streams and assign them to speakers by cosine similarity to the anchors (straight vs. swapped pairing, whichever maximizes sum-sim).
4. Concatenate per speaker in time order: solo-enhanced + assigned-overlap, with a 0.3 s silence separator between pieces. The resulting `timestamp_map` records the mapping from concat-time to original-recording time, so word timestamps can be mapped back to the source later.

In [ ]:
@torch.no_grad()
def _ecapa_embed(audio_16k: np.ndarray) -> torch.Tensor:
    """Returns a unit-norm ECAPA embedding. Caller ensures `ecapa` is on DEVICE."""
    if len(audio_16k) < SR_PIPELINE // 4:
        audio_16k = np.pad(audio_16k, (0, SR_PIPELINE // 4 - len(audio_16k)))
    audio = torch.from_numpy(audio_16k).unsqueeze(0).to(DEVICE)
    emb = ecapa.encode_batch(audio).squeeze(0).squeeze(0)
    emb = emb / (emb.norm() + 1e-8)
    return emb


def _cos(a, b):
    return float((a * b).sum())


def assemble(solo_enhanced, overlap_separated, speakers):
    """Build one per-speaker concatenated stream."""
    spk_to_label = {spk: chr(ord("A") + i) for i, spk in enumerate(speakers)}

    solo_by_spk = {spk: [] for spk in speakers}
    for seg in sorted(solo_enhanced.values(), key=lambda x: x["start"]):
        if seg["speaker"] in solo_by_spk:
            solo_by_spk[seg["speaker"]].append(seg)

    # All ECAPA work runs under one on_gpu(ecapa) block.
    anchors = {}
    solo_durations = {}
    assignments = []

    with on_gpu(ecapa):
        for spk in speakers:
            concat = np.concatenate(
                [s["enhanced"] for s in solo_by_spk[spk]] or [np.zeros(16, dtype=np.float32)]
            )
            solo_durations[spk] = len(concat) / SR_PIPELINE
            anchors[spk] = (
                _ecapa_embed(concat).cpu() if len(concat) >= SR_PIPELINE // 4 else None
            )
        weak_anchor = any(d < MIN_SOLO_FOR_ANCHOR for d in solo_durations.values())

        for ovl in overlap_separated:
            if len(ovl["s1_gated"]) < SR_PIPELINE // 10:
                continue
            emb1 = _ecapa_embed(ovl["s1_gated"]).cpu()
            emb2 = _ecapa_embed(ovl["s2_gated"]).cpu()
            if all(anchors.get(s) is not None for s in speakers):
                a, b = speakers[0], speakers[1]
                straight = _cos(emb1, anchors[a]) + _cos(emb2, anchors[b])
                swapped  = _cos(emb1, anchors[b]) + _cos(emb2, anchors[a])
                if straight >= swapped:
                    stream_for = {a: ovl["s1_gated"], b: ovl["s2_gated"]}
                    chosen = "straight"
                else:
                    stream_for = {a: ovl["s2_gated"], b: ovl["s1_gated"]}
                    chosen = "swapped"
            else:
                stream_for = {speakers[0]: ovl["s1_gated"], speakers[1]: ovl["s2_gated"]}
                chosen = "arbitrary (weak anchor)"
            assignments.append({
                "start": ovl["start"], "end": ovl["end"],
                "stream_for": stream_for, "pairing": chosen,
            })

    GAP = int(0.3 * SR_PIPELINE)
    events = []
    for spk in speakers:
        for seg in solo_by_spk[spk]:
            events.append((seg["start"], seg["end"], spk, seg["enhanced"], "solo"))
    for a in assignments:
        for spk, audio in a["stream_for"].items():
            events.append((a["start"], a["end"], spk, audio, "overlap"))

    assembled = {}
    timestamp_map = {}
    for spk in speakers:
        pieces = []
        tmap = []
        cursor = 0.0
        for ev in sorted((e for e in events if e[2] == spk), key=lambda x: x[0]):
            if pieces:
                pieces.append(np.zeros(GAP, dtype=np.float32))
                cursor += GAP / SR_PIPELINE
            audio = ev[3].astype(np.float32)
            pieces.append(audio)
            dur = len(audio) / SR_PIPELINE
            tmap.append((cursor, cursor + dur, ev[0], ev[1], ev[4]))
            cursor += dur
        assembled[spk] = (
            np.concatenate(pieces) if pieces else np.zeros(SR_PIPELINE, dtype=np.float32)
        )
        timestamp_map[spk] = tmap

    return assembled, anchors, assignments, weak_anchor, timestamp_map, spk_to_label


def plot_assembly(timestamp_map, total_dur, spk_to_label):
    """Per-speaker timeline strip: blue = solo/SE, red = overlap/SS."""
    fig, axes = plt.subplots(len(timestamp_map), 1,
                             figsize=(16, 1.2 * len(timestamp_map) + 1),
                             sharex=True, squeeze=False)
    axes = axes[:, 0]
    for ax, (spk, tmap) in zip(axes, timestamp_map.items()):
        for (_, _, orig_s, orig_e, kind) in tmap:
            color = "#1f77b4" if kind == "solo" else "#e74c3c"
            ax.barh(0, orig_e - orig_s, left=orig_s, height=0.6,
                    color=color, alpha=0.85)
        ax.set_yticks([0])
        ax.set_yticklabels([f"{spk_to_label[spk]}: {spk}"])
        ax.set_ylim(-0.5, 0.5)
        ax.grid(axis="x", alpha=0.3)
    axes[-1].set_xlabel("Time (s)")
    axes[-1].set_xlim(0, total_dur)
    axes[0].set_title("Assembled streams  (blue = solo/SE, red = overlap/SS)")
    plt.tight_layout()
    return fig

## Stage 5 — Whisper ASR

One Whisper call per assembled per-speaker stream. `language='pl'` is forced and `initial_prompt` is a short Polish sentence to discourage language drift on noisy tails. Word-level timestamps stay enabled so the per-segment view remains tractable when intrusive metrics are added later.

In [ ]:
def transcribe(assembled):
    """Whisper one call per per-speaker stream. Returns dict[spk, whisper_result]."""
    out = {}
    if not assembled:
        return out
    with on_gpu(whisper_model):
        for spk, audio in assembled.items():
            audio32 = audio.astype(np.float32)
            if len(audio32) < SR_PIPELINE // 2:
                out[spk] = {"text": "", "segments": []}
                continue
            result = whisper_model.transcribe(
                audio32, language="pl",
                initial_prompt=WHISPER_INITIAL_PROMPT,
                word_timestamps=True, verbose=False,
            )
            out[spk] = result
    return out


def format_transcript(whisper_result) -> str:
    segments = whisper_result.get("segments", []) if isinstance(whisper_result, dict) else []
    if not segments:
        return whisper_result.get("text", "") or "(empty)"
    return "\n".join(
        f"[{seg['start']:6.2f} → {seg['end']:6.2f}]  {seg['text'].strip()}"
        for seg in segments
    )

## Stage 6 — SQUIM non-intrusive quality

`SQUIM_OBJECTIVE` returns predicted STOI, PESQ, SI-SDR per audio. We report it for the raw mixture and for each per-speaker assembled stream so the *relative* change is visible. SQUIM is trained on English so absolute values on Polish audio are non-diagnostic — only the mixture-vs-assembled deltas matter for sanity-checking the pipeline.

In [ ]:
@torch.no_grad()
def _squim_one(audio: np.ndarray):
    if len(audio) < SR_PIPELINE:
        audio = np.pad(audio, (0, SR_PIPELINE - len(audio)))
    t = torch.from_numpy(audio.astype(np.float32)).unsqueeze(0).to(DEVICE)
    stoi, pesq, sisdr = squim_obj(t)
    return float(stoi.item()), float(pesq.item()), float(sisdr.item())


def squim_metrics(mix, assembled, spk_to_label):
    rows = []
    with on_gpu(squim_obj):
        s, p, sd = _squim_one(mix)
        rows.append({"stream": "mixture",
                     "STOI": round(s, 3), "PESQ": round(p, 2), "SI-SDR (dB)": round(sd, 2)})
        for spk, audio in assembled.items():
            s, p, sd = _squim_one(audio)
            rows.append({"stream": f"assembled-{spk_to_label[spk]} ({spk})",
                         "STOI": round(s, 3), "PESQ": round(p, 2), "SI-SDR (dB)": round(sd, 2)})
    return pd.DataFrame(rows)

## Gradio UI

One drag-and-drop input, sliders for the routing/VAD thresholds, one **Run pipeline** button. Each stage's output lands in its own accordion section so you can expand whichever stage you want to inspect.

For Stage 3a / 3b the visualization shows only the *first* segment of its kind (a full per-segment dropdown would significantly complicate the Gradio plumbing — kept out of scope for v1). The audio playback widgets at the bottom let you listen to each fully assembled per-speaker stream.

In [ ]:
import gradio as gr


def _audio_for_gradio(arr: np.ndarray):
    """Gradio expects int16 or float32 in (-1, 1). Clip and cast."""
    arr = np.clip(arr.astype(np.float32), -1.0, 1.0)
    return (SR_PIPELINE, arr)


def run_pipeline(audio_path, min_overlap, max_gap, pad, vad_th):
    if audio_path is None:
        raise gr.Error("Upload an audio file first.")

    audio_16k = load_audio_as_mono_16k(audio_path)

    # Stage 1
    seg_df, ovl_df, dur = diarize(audio_16k)
    fig1 = plot_diarization(seg_df, ovl_df, dur, "Stage 1 — Diarization")

    # Stage 2
    part_df, speakers = route(seg_df, ovl_df, dur, min_overlap, max_gap, pad)
    fig2 = plot_routing(seg_df, ovl_df, part_df, dur)

    # Stage 3a
    solo_enh = enhance_solo(audio_16k, part_df)
    if solo_enh:
        first_solo = sorted(solo_enh.values(), key=lambda x: x["start"])[0]
        fig3a = plot_se_segment(first_solo)
    else:
        fig3a, _a = plt.subplots(figsize=(8, 2)); _a.text(0.5, 0.5, "No solo segments",
                                                          ha="center", va="center")

    # Stage 3b
    ovl_sep = separate_and_vad(audio_16k, part_df, vad_th)
    if ovl_sep:
        fig3b = plot_overlap_segment(ovl_sep[0])
    else:
        fig3b, _a = plt.subplots(figsize=(8, 2)); _a.text(0.5, 0.5, "No overlap segments",
                                                          ha="center", va="center")

    # Stage 4
    if speakers:
        assembled, anchors, assignments, weak, ts_map, spk_to_label = assemble(
            solo_enh, ovl_sep, speakers
        )
        fig4 = plot_assembly(ts_map, dur, spk_to_label)
    else:
        assembled, weak, spk_to_label = {}, False, {}
        fig4, _a = plt.subplots(figsize=(8, 2)); _a.text(0.5, 0.5, "No speakers",
                                                         ha="center", va="center")

    # Stage 5
    transcripts = transcribe(assembled) if assembled else {}
    label_to_spk = {v: k for k, v in spk_to_label.items()}
    text_a = format_transcript(transcripts.get(label_to_spk.get("A", "_missing"), {})) \
        if "A" in label_to_spk else "(no speaker A)"
    text_b = format_transcript(transcripts.get(label_to_spk.get("B", "_missing"), {})) \
        if "B" in label_to_spk else "(no speaker B)"

    audio_a = _audio_for_gradio(assembled[label_to_spk["A"]]) if "A" in label_to_spk else None
    audio_b = _audio_for_gradio(assembled[label_to_spk["B"]]) if "B" in label_to_spk else None

    # Stage 6
    metrics_df = squim_metrics(audio_16k, assembled, spk_to_label) if assembled else \
        pd.DataFrame(columns=["stream", "STOI", "PESQ", "SI-SDR (dB)"])

    counts = part_df["kind"].value_counts().to_dict() if len(part_df) else {}
    status = (
        f"**Duration:** {dur:.1f}s | **speakers:** {len(speakers)} | "
        f"**routing:** silence={counts.get('silence', 0)}, "
        f"solo-A={counts.get('solo-A', 0)}, solo-B={counts.get('solo-B', 0)}, "
        f"overlap={counts.get('overlap', 0)} | **weak_anchor:** {weak}"
    )

    return (
        status,
        fig1, seg_df, ovl_df,
        fig2, part_df,
        fig3a,
        fig3b,
        fig4,
        text_a, text_b,
        audio_a, audio_b,
        metrics_df,
    )


with gr.Blocks(title="ASR Pipeline POC", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# ASR Pipeline POC\nDrag and drop an audio file, then click **Run pipeline**.")

    with gr.Row():
        audio_input = gr.Audio(type="filepath", label="Audio")
        with gr.Column():
            min_overlap_slider = gr.Slider(0.0, 1.0, value=MIN_OVERLAP_DUR, step=0.05,
                                           label="Min overlap duration (s) — shorter dropped")
            max_gap_slider = gr.Slider(0.0, 0.5, value=MAX_MERGE_GAP, step=0.02,
                                       label="Max merge gap (s) — same-speaker merging")
            pad_slider = gr.Slider(0.0, 0.5, value=SEGMENT_PAD, step=0.02,
                                   label="Segment pad (s)")
            vad_slider = gr.Slider(0.0, 1.0, value=VAD_THRESHOLD, step=0.05,
                                   label="VAD threshold")

    run_btn = gr.Button("Run pipeline", variant="primary", size="lg")
    status_md = gr.Markdown()

    with gr.Accordion("Stage 1 — Diarization", open=False):
        s1_plot = gr.Plot()
        with gr.Row():
            s1_segs = gr.Dataframe(label="Segments", wrap=True)
            s1_ovls = gr.Dataframe(label="Overlaps", wrap=True)

    with gr.Accordion("Stage 2 — Routing", open=False):
        s2_plot = gr.Plot()
        s2_table = gr.Dataframe(label="Partition", wrap=True)

    with gr.Accordion("Stage 3a — SE on solo regions (first segment shown)", open=False):
        s3a_plot = gr.Plot()

    with gr.Accordion("Stage 3b — SS + VAD on overlap regions (first segment shown)", open=False):
        s3b_plot = gr.Plot()

    with gr.Accordion("Stage 4 — Assembly", open=False):
        s4_plot = gr.Plot()

    with gr.Accordion("Stage 5 — ASR transcripts", open=True):
        with gr.Row():
            t_a = gr.Textbox(label="Speaker A", lines=20)
            t_b = gr.Textbox(label="Speaker B", lines=20)
        with gr.Row():
            a_a = gr.Audio(label="Assembled stream A", type="numpy")
            a_b = gr.Audio(label="Assembled stream B", type="numpy")

    with gr.Accordion("Stage 6 — SQUIM non-intrusive quality", open=True):
        s6_table = gr.Dataframe(label="SQUIM metrics")

    run_btn.click(
        fn=run_pipeline,
        inputs=[audio_input, min_overlap_slider, max_gap_slider, pad_slider, vad_slider],
        outputs=[
            status_md,
            s1_plot, s1_segs, s1_ovls,
            s2_plot, s2_table,
            s3a_plot,
            s3b_plot,
            s4_plot,
            t_a, t_b,
            a_a, a_b,
            s6_table,
        ],
    )

demo.launch(inline=True, share=False)